<div style="font-size: 1em; display: flex; align-items: center; gap: 8px; padding: 8px 16px; background: #F8F9FA; border-bottom: 2px solid #E0E0E0; margin: 0; line-height: 1">
    <img src="https://cdn.simpleicons.org/databricks/FF3621" width="24" height="24"/>
    <div style="color: #666">
        <span style="font-weight: bold; color: #333">Introduction to Apache Spark</span>
        <span style="margin-left: 8px; color: #999">|</span>
        <span style="margin-left: 8px">3. Distributed Systems Programming Fundamentals</span>
    </div>
</div>

<p style="font-size: 1em; text-align: center; line-height: 0; padding-top: 9px; margin: 4px 0">
<img
src="https://databricks.com/wp-content/uploads/2018/03/db-academy-rgb-1200px.png"
alt="Databricks Learning"
>
</p>

# 3.1 Lecture - Distributed Systems Programming Fundamentals
### Duration - 12 Minutes

> *How Spark divides data and moves it between machines*

This lecture covers the distributed systems concepts behind Spark's execution model. Understanding partitioning and the shuffle explains both why Spark scales and where its costs come from.

## Learning Objectives

**By the end of this lecture, you will be able to:**

- Describe core distributed system concepts including shared-nothing architecture, partitioning, and parallel processing
- Explain how data moves between nodes in distributed operations, particularly during shuffle operations like `groupBy` and `join`
- Describe the MapReduce algorithm for parallel data processing and its application in the DataFrame API

## A. Shared Nothing Architecture

<p style="font-size: 1em; line-height: 1.6; color: #333">Spark follows the <strong>shared nothing</strong> model of distributed computing: nodes cooperate on a job without sharing memory or disk.</p>

<div style="display: grid; grid-template-columns: 1fr 1fr; gap: 12px; margin: 16px 0">
<div style="background: #F9F7F4; border: 1px solid #DCE0E2; border-radius: 10px; padding: 18px">
<div style="font-weight: bold; color: #1B5162; margin-bottom: 8px">Independence</div>
<div style="color: #333">Each node works independently and manages its own resources (CPU, memory, disk).</div>
</div>
<div style="background: #F9F7F4; border: 1px solid #DCE0E2; border-radius: 10px; padding: 18px">
<div style="font-weight: bold; color: #1B5162; margin-bottom: 8px">Scalability</div>
<div style="color: #333">Adding more nodes can improve performance without resource contention.</div>
</div>
<div style="background: #F9F7F4; border: 1px solid #DCE0E2; border-radius: 10px; padding: 18px">
<div style="font-weight: bold; color: #1B5162; margin-bottom: 8px">Fault Tolerance</div>
<div style="color: #333">Failures are isolated to individual nodes, minimizing system-wide impacts.</div>
</div>
<div style="background: #F9F7F4; border: 1px solid #DCE0E2; border-radius: 10px; padding: 18px">
<div style="font-weight: bold; color: #1B5162; margin-bottom: 8px">Resource Partitioning</div>
<div style="color: #333">Data and workloads are partitioned across nodes to eliminate contention.</div>
</div>
</div>

## B. Partitioning

<p style="font-size: 1em; line-height: 1.6; color: #333">Partitioning is how Spark distributes data across the cluster.</p>

**Data distribution:**

- Data is divided into mutually exclusive in-memory partitions
- Default partitioning is based upon the input and can be manipulated
- The size and number of partitions affect parallelism

**Processing model:**

- Each partition is processed independently
- Multiple partitions can run in parallel
- One partition = one task in Spark
- Partitioning impacts performance and shuffle operations

<details>
  <summary style="cursor: pointer; list-style: none; user-select: none">
    <div style="border-left: 4px solid #1B5162; background: transparent; padding: 16px 20px; border-radius: 4px; margin: 16px 0">
      <div style="display: flex; align-items: center; gap: 12px">
        <span>&#x25B6;</span>
        <strong style="font-size: 1.1em; color: #1B5162">Partitions distributed across Workers</strong>
      </div>
    </div>
  </summary>
  <div style="width:100%; max-width:820px; margin:8px 0">
  <svg viewBox="0 0 760 320" width="100%" style="height:auto; font-family:-apple-system,BlinkMacSystemFont,'Segoe UI',Roboto,sans-serif">
    <defs><marker id="pt-arrow" markerWidth="10" markerHeight="8" refX="7" refY="3" orient="auto" markerUnits="strokeWidth"><path d="M0,0 L8,3 L0,6 Z" fill="#618794"/></marker></defs>
    <path d="M20,105 A55,12 0 0 1 130,105 L130,195 A55,12 0 0 1 20,195 Z" fill="#DCE0E2" stroke="#618794" stroke-width="2"/>
    <ellipse cx="75" cy="105" rx="55" ry="12" fill="#DCE0E2" stroke="#618794" stroke-width="2"/>
    <text x="75" y="146" text-anchor="middle" font-size="15" font-weight="700" fill="#1B5162">Input</text>
    <text x="75" y="166" text-anchor="middle" font-size="15" font-weight="700" fill="#1B5162">Data Set</text>
    <line x1="134" y1="150" x2="366" y2="59" stroke="#618794" stroke-width="2" marker-end="url(#pt-arrow)"/>
    <line x1="134" y1="150" x2="366" y2="160" stroke="#618794" stroke-width="2" marker-end="url(#pt-arrow)"/>
    <line x1="134" y1="150" x2="366" y2="261" stroke="#618794" stroke-width="2" marker-end="url(#pt-arrow)"/>
    <g><rect x="370" y="16" width="374" height="86" rx="8" fill="#FFFFFF" stroke="#DCE0E2" stroke-width="1.5"/><text x="384" y="36" font-size="13" fill="#618794">Worker</text><rect x="384" y="44" width="346" height="48" rx="6" fill="#F9F7F4" stroke="#DCE0E2" stroke-width="1.5"/><text x="398" y="62" font-size="12" fill="#618794">Executor</text><rect x="560" y="50" width="160" height="34" rx="6" fill="#FFFFFF" stroke="#618794" stroke-width="1.5"/><text x="640" y="72" text-anchor="middle" font-size="14" fill="#333">Partition 0</text></g>
    <g><rect x="370" y="117" width="374" height="86" rx="8" fill="#FFFFFF" stroke="#DCE0E2" stroke-width="1.5"/><text x="384" y="137" font-size="13" fill="#618794">Worker</text><rect x="384" y="145" width="346" height="48" rx="6" fill="#F9F7F4" stroke="#DCE0E2" stroke-width="1.5"/><text x="398" y="163" font-size="12" fill="#618794">Executor</text><rect x="560" y="151" width="160" height="34" rx="6" fill="#FFFFFF" stroke="#618794" stroke-width="1.5"/><text x="640" y="173" text-anchor="middle" font-size="14" fill="#333">Partition 1</text></g>
    <g><rect x="370" y="218" width="374" height="86" rx="8" fill="#FFFFFF" stroke="#DCE0E2" stroke-width="1.5"/><text x="384" y="238" font-size="13" fill="#618794">Worker</text><rect x="384" y="246" width="346" height="48" rx="6" fill="#F9F7F4" stroke="#DCE0E2" stroke-width="1.5"/><text x="398" y="264" font-size="12" fill="#618794">Executor</text><rect x="560" y="252" width="160" height="34" rx="6" fill="#FFFFFF" stroke="#618794" stroke-width="1.5"/><text x="640" y="274" text-anchor="middle" font-size="14" fill="#333">Partition 2</text></g>
  </svg>
  </div>
</details>

## C. The Shuffle Operation

<p style="font-size: 1em; line-height: 1.6; color: #333">The shuffle is how data moves between partitions in distributed processing.</p>

**What is shuffling?**

- Redistributing data across partitions
- Required for operations like `groupBy` and `join`
- The most expensive operation in Spark - it involves serializing data and moving it over the network

**When it occurs:**

- Wide transformations (where output partitions depend on many input partitions)
- Key-based operations
- Data repartitioning

<details>
  <summary style="cursor: pointer; list-style: none; user-select: none">
    <div style="border-left: 4px solid #1B5162; background: transparent; padding: 16px 20px; border-radius: 4px; margin: 16px 0">
      <div style="display: flex; align-items: center; gap: 12px">
        <span>&#x25B6;</span>
        <strong style="font-size: 1.1em; color: #1B5162">All-to-all data movement between stages</strong>
      </div>
    </div>
  </summary>
  <div style="width:100%; max-width:900px; margin:8px 0">
  <svg viewBox="0 0 820 340" width="100%" style="height:auto; font-family:-apple-system,BlinkMacSystemFont,'Segoe UI',Roboto,sans-serif">
    <defs><marker id="shuf-arrow" markerWidth="10" markerHeight="8" refX="7" refY="3" orient="auto" markerUnits="strokeWidth"><path d="M0,0 L8,3 L0,6 Z" fill="#618794"/></marker></defs>
    <rect x="16" y="16" width="300" height="308" rx="10" fill="#F9F7F4" stroke="#618794" stroke-width="2"/>
    <text x="166" y="42" text-anchor="middle" font-size="15" font-weight="700" fill="#1B5162">Stage 1 - map side</text>
    <rect x="41" y="64" width="250" height="64" rx="8" fill="#FFFFFF" stroke="#DCE0E2" stroke-width="1.5"/><text x="166" y="92" text-anchor="middle" font-size="14" font-weight="600" fill="#333">partition 0</text><text x="166" y="114" text-anchor="middle" font-size="12" fill="#555">keys k1, k2</text>
    <rect x="41" y="148" width="250" height="64" rx="8" fill="#FFFFFF" stroke="#DCE0E2" stroke-width="1.5"/><text x="166" y="176" text-anchor="middle" font-size="14" font-weight="600" fill="#333">partition 1</text><text x="166" y="198" text-anchor="middle" font-size="12" fill="#555">keys k1, k2</text>
    <rect x="41" y="232" width="250" height="64" rx="8" fill="#FFFFFF" stroke="#DCE0E2" stroke-width="1.5"/><text x="166" y="260" text-anchor="middle" font-size="14" font-weight="600" fill="#333">partition 2</text><text x="166" y="282" text-anchor="middle" font-size="12" fill="#555">keys k1, k2</text>
    <rect x="504" y="60" width="300" height="220" rx="10" fill="#FFCC66" stroke="#CD7F32" stroke-width="2"/>
    <text x="654" y="86" text-anchor="middle" font-size="15" font-weight="700" fill="#7a4a10">Stage 2 - reduce side</text>
    <rect x="529" y="104" width="250" height="64" rx="8" fill="#FFFFFF" stroke="#CD7F32" stroke-width="1.5"/><text x="654" y="132" text-anchor="middle" font-size="14" font-weight="600" fill="#333">partition 0</text><text x="654" y="154" text-anchor="middle" font-size="12" fill="#555">all k1 rows</text>
    <rect x="529" y="192" width="250" height="64" rx="8" fill="#FFFFFF" stroke="#CD7F32" stroke-width="1.5"/><text x="654" y="220" text-anchor="middle" font-size="14" font-weight="600" fill="#333">partition 1</text><text x="654" y="242" text-anchor="middle" font-size="12" fill="#555">all k2 rows</text>
    <line x1="293" y1="90" x2="527" y2="130" stroke="#618794" stroke-width="1.5" marker-end="url(#shuf-arrow)"/>
    <line x1="293" y1="104" x2="527" y2="216" stroke="#618794" stroke-width="1.5" marker-end="url(#shuf-arrow)"/>
    <line x1="293" y1="174" x2="527" y2="136" stroke="#618794" stroke-width="1.5" marker-end="url(#shuf-arrow)"/>
    <line x1="293" y1="188" x2="527" y2="222" stroke="#618794" stroke-width="1.5" marker-end="url(#shuf-arrow)"/>
    <line x1="293" y1="256" x2="527" y2="142" stroke="#618794" stroke-width="1.5" marker-end="url(#shuf-arrow)"/>
    <line x1="293" y1="270" x2="527" y2="228" stroke="#618794" stroke-width="1.5" marker-end="url(#shuf-arrow)"/>
    <rect x="299" y="79" width="20" height="15" fill="#FFFFFF"/><text x="309" y="91" text-anchor="middle" font-size="11" fill="#1B5162">k1</text>
    <rect x="299" y="99" width="20" height="15" fill="#FFFFFF"/><text x="309" y="111" text-anchor="middle" font-size="11" fill="#1B5162">k2</text>
    <rect x="299" y="163" width="20" height="15" fill="#FFFFFF"/><text x="309" y="175" text-anchor="middle" font-size="11" fill="#1B5162">k1</text>
    <rect x="299" y="183" width="20" height="15" fill="#FFFFFF"/><text x="309" y="195" text-anchor="middle" font-size="11" fill="#1B5162">k2</text>
    <rect x="299" y="245" width="20" height="15" fill="#FFFFFF"/><text x="309" y="257" text-anchor="middle" font-size="11" fill="#1B5162">k1</text>
    <rect x="299" y="265" width="20" height="15" fill="#FFFFFF"/><text x="309" y="277" text-anchor="middle" font-size="11" fill="#1B5162">k2</text>
  </svg>
  </div>
</details>
<br/>
<details>
<summary style="cursor: pointer; list-style: none; user-select: none">
<div style="border-left: 4px solid #1B5162; background: transparent; padding: 16px 20px; border-radius: 4px; margin: 16px 0">
<div style="display: flex; align-items: center; gap: 12px">
<span>&#x25B6;</span>
<strong style="font-size: 1.1em; color: #1B5162">Expand for More Details</strong>
</div>
</div>
</summary>
<div style="border-left: 4px solid #1B5162; padding: 4px 24px 14px 24px; margin: -16px 0 16px 0">
<div style="font-size: 1.2em; font-weight: bold; color: #1B5162; margin: 12px 0 4px 0">Why the Shuffle is Expensive</div>
<ul style="font-size: 1.1em; line-height: 1.45; color: #333; margin: 0 0 10px 0; padding-left: 24px">
<li style="margin: 4px 0">Every map-side partition may hold rows for every key, so each reduce-side partition must collect data from <em>all</em> map-side partitions - an all-to-all exchange across the network.</li>
<li style="margin: 4px 0">Shuffled data is serialized, written to disk on the map side, transferred, and deserialized on the reduce side. This is why a stage boundary appears at every shuffle in the DAG.</li>
<li style="margin: 4px 0">Minimizing shuffles - by filtering early and aggregating as late as possible with as little data as possible - is the single most effective Spark performance habit.</li>
</ul>
<div style="font-size: 1.2em; font-weight: bold; color: #1B5162; margin: 14px 0 4px 0">Adaptive Query Execution Helps</div>
<ul style="font-size: 1.1em; line-height: 1.45; color: #333; margin: 0 0 10px 0; padding-left: 24px">
<li style="margin: 4px 0">Since Spark 3 (and on by default in Spark 4), Adaptive Query Execution inspects the actual shuffle output at runtime and coalesces small shuffle partitions, picks better join strategies, and rebalances skewed partitions automatically.</li>
</ul>
</div>
</details>

## D. MapReduce in Action

<p style="font-size: 1em; line-height: 1.6; color: #333">MapReduce is the classic pattern for parallel data processing, and it is exactly what a DataFrame pipeline compiles down to. Consider this small ETL job, which reads sales records, filters them, and totals the amount per id:</p>

<pre style="background:#1e1e1e;color:#cccccc;border-radius:8px;padding:16px;margin:16px 0;overflow-x:auto;font-family:Consolas,Monaco,monospace;font-size:14px;line-height:1.5"><code><span style="color: #999"># READ: scan the source file into a DataFrame</span>
<span style="color: #CCC">input_df</span> <span style="color: #67CDCC">=</span> <span style="color: #CCC">spark</span><span style="color: #67CDCC">.</span><span style="color: #CCC">read</span><span style="color: #67CDCC">.</span><span style="color: #CCC">csv(</span><span style="color: #7EC699">"sales_data.csv"</span><span style="color: #CCC">)</span>
<span></span>
<span style="color: #999"># MAP: narrow, per-partition transformations - no data movement</span>
<span style="color: #CCC">filtered_df</span> <span style="color: #67CDCC">=</span> <span style="color: #CCC">input_df</span><span style="color: #67CDCC">.</span><span style="color: #CCC">select(</span><span style="color: #7EC699">"id"</span><span style="color: #CCC">,</span> <span style="color: #7EC699">"amt"</span><span style="color: #CCC">)</span><span style="color: #67CDCC">.</span><span style="color: #CCC">filter(</span><span style="color: #7EC699">"amt &gt; 50"</span><span style="color: #CCC">)</span>
<span></span>
<span style="color: #999"># SHUFFLE: groupBy sets the key, so all rows for the same id must be</span>
<span style="color: #999"># moved onto the same partition (the exchange) before aggregation</span>
<span style="color: #CCC">grouped_df</span> <span style="color: #67CDCC">=</span> <span style="color: #CCC">filtered_df</span><span style="color: #67CDCC">.</span><span style="color: #CCC">groupBy(</span><span style="color: #7EC699">"id"</span><span style="color: #CCC">)</span>
<span></span>
<span style="color: #999"># REDUCE: aggregate each group after the shuffle - sum amt per id</span>
<span style="color: #CCC">aggregated_df</span> <span style="color: #67CDCC">=</span> <span style="color: #CCC">grouped_df</span><span style="color: #67CDCC">.</span><span style="color: #CCC">agg({</span><span style="color: #7EC699">"amt"</span><span style="color: #CCC">:</span> <span style="color: #7EC699">"sum"</span><span style="color: #CCC">})</span>
<span></span>
<span style="color: #999"># WRITE: each partition writes its slice of the result</span>
<span style="color: #CCC">aggregated_df</span><span style="color: #67CDCC">.</span><span style="color: #CCC">write</span><span style="color: #67CDCC">.</span><span style="color: #CCC">csv(</span><span style="color: #7EC699">"/Volumes/catalog/schema/output/sales_by_id"</span><span style="color: #CCC">)</span></code></pre>

<p style="font-size: 1em; line-height: 1.6; color: #333">Click the bar below to trace the data through the map stage, the shuffle, and the reduce stage.</p>

<details>
  <summary style="cursor: pointer; list-style: none; user-select: none">
    <div style="border-left: 4px solid #1B5162; background: transparent; padding: 16px 20px; border-radius: 4px; margin: 16px 0">
      <div style="display: flex; align-items: center; gap: 12px">
        <span>&#x25B6;</span>
        <strong style="font-size: 1.1em; color: #1B5162">MapReduce data flow for the sales pipeline</strong>
      </div>
    </div>
  </summary>
  <div style="width:100%; max-width:820px; margin:8px 0">
  <svg viewBox="0 0 800 766" width="100%" style="height:auto; font-family:-apple-system,BlinkMacSystemFont,'Segoe UI',Roboto,sans-serif">
    <defs><marker id="mr-arrow" markerWidth="10" markerHeight="8" refX="7" refY="3" orient="auto" markerUnits="strokeWidth"><path d="M0,0 L8,3 L0,6 Z" fill="#618794"/></marker></defs>
    <g font-family="Consolas,Monaco,monospace">
    <g><rect x="16" y="16" width="180" height="110" rx="8" fill="#FFFFFF" stroke="#DCE0E2" stroke-width="1.5"/><text x="106" y="38" text-anchor="middle" font-size="14" font-weight="700" fill="#1B5162" font-family="-apple-system,sans-serif">file 1</text><text x="106" y="58" text-anchor="middle" font-size="11" fill="#888">id,amt,region</text><text x="106" y="78" text-anchor="middle" font-size="12" fill="#333">A,100,US</text><text x="106" y="96" text-anchor="middle" font-size="12" fill="#333">C,200,US</text><text x="106" y="114" text-anchor="middle" font-size="12" fill="#999">B,30,EU</text></g>
    <g><rect x="212" y="16" width="180" height="110" rx="8" fill="#FFFFFF" stroke="#DCE0E2" stroke-width="1.5"/><text x="302" y="38" text-anchor="middle" font-size="14" font-weight="700" fill="#1B5162" font-family="-apple-system,sans-serif">file 2</text><text x="302" y="58" text-anchor="middle" font-size="11" fill="#888">id,amt,region</text><text x="302" y="78" text-anchor="middle" font-size="12" fill="#333">C,60,EU</text><text x="302" y="96" text-anchor="middle" font-size="12" fill="#333">D,150,EU</text><text x="302" y="114" text-anchor="middle" font-size="12" fill="#333">B,75,US</text></g>
    <g><rect x="408" y="16" width="180" height="110" rx="8" fill="#FFFFFF" stroke="#DCE0E2" stroke-width="1.5"/><text x="498" y="38" text-anchor="middle" font-size="14" font-weight="700" fill="#1B5162" font-family="-apple-system,sans-serif">file 3</text><text x="498" y="58" text-anchor="middle" font-size="11" fill="#888">id,amt,region</text><text x="498" y="78" text-anchor="middle" font-size="12" fill="#333">A,120,US</text><text x="498" y="96" text-anchor="middle" font-size="12" fill="#333">A,90,EU</text><text x="498" y="114" text-anchor="middle" font-size="12" fill="#999">D,50,EU</text></g>
    <g><rect x="604" y="16" width="180" height="110" rx="8" fill="#FFFFFF" stroke="#DCE0E2" stroke-width="1.5"/><text x="694" y="38" text-anchor="middle" font-size="14" font-weight="700" fill="#1B5162" font-family="-apple-system,sans-serif">file 4</text><text x="694" y="58" text-anchor="middle" font-size="11" fill="#888">id,amt,region</text><text x="694" y="78" text-anchor="middle" font-size="12" fill="#333">C,80,US</text><text x="694" y="96" text-anchor="middle" font-size="12" fill="#333">C,140,EU</text><text x="694" y="114" text-anchor="middle" font-size="12" fill="#333">B,110,US</text></g>
    <line x1="106" y1="126" x2="106" y2="154" stroke="#618794" stroke-width="2" marker-end="url(#mr-arrow)"/><line x1="302" y1="126" x2="302" y2="154" stroke="#618794" stroke-width="2" marker-end="url(#mr-arrow)"/><line x1="498" y1="126" x2="498" y2="154" stroke="#618794" stroke-width="2" marker-end="url(#mr-arrow)"/><line x1="694" y1="126" x2="694" y2="154" stroke="#618794" stroke-width="2" marker-end="url(#mr-arrow)"/>
    <rect x="16" y="156" width="768" height="64" rx="8" fill="#FABFBA" stroke="#CD7F32" stroke-width="2"/><text x="400" y="183" text-anchor="middle" font-size="14" font-weight="700" fill="#7a2018" font-family="-apple-system,sans-serif">Map stage: select('id', 'amt') + filter('amt &gt; 50')</text><text x="400" y="204" text-anchor="middle" font-size="12" font-style="italic" fill="#8a4038" font-family="-apple-system,sans-serif">projection and filtering are narrow transformations</text>
    <line x1="106" y1="220" x2="106" y2="248" stroke="#618794" stroke-width="2" marker-end="url(#mr-arrow)"/><line x1="302" y1="220" x2="302" y2="248" stroke="#618794" stroke-width="2" marker-end="url(#mr-arrow)"/><line x1="498" y1="220" x2="498" y2="248" stroke="#618794" stroke-width="2" marker-end="url(#mr-arrow)"/><line x1="694" y1="220" x2="694" y2="248" stroke="#618794" stroke-width="2" marker-end="url(#mr-arrow)"/>
    <g><rect x="16" y="250" width="180" height="88" rx="8" fill="#FFFFFF" stroke="#DCE0E2" stroke-width="1.5"/><text x="106" y="272" text-anchor="middle" font-size="13" font-weight="700" fill="#1B5162" font-family="-apple-system,sans-serif">part0</text><text x="106" y="294" text-anchor="middle" font-size="12" fill="#333">A,100</text><text x="106" y="312" text-anchor="middle" font-size="12" fill="#333">C,200</text></g>
    <g><rect x="212" y="250" width="180" height="88" rx="8" fill="#FFFFFF" stroke="#DCE0E2" stroke-width="1.5"/><text x="302" y="272" text-anchor="middle" font-size="13" font-weight="700" fill="#1B5162" font-family="-apple-system,sans-serif">part1</text><text x="302" y="294" text-anchor="middle" font-size="12" fill="#333">C,60</text><text x="302" y="311" text-anchor="middle" font-size="12" fill="#333">D,150</text><text x="302" y="328" text-anchor="middle" font-size="12" fill="#333">B,75</text></g>
    <g><rect x="408" y="250" width="180" height="88" rx="8" fill="#FFFFFF" stroke="#DCE0E2" stroke-width="1.5"/><text x="498" y="272" text-anchor="middle" font-size="13" font-weight="700" fill="#1B5162" font-family="-apple-system,sans-serif">part2</text><text x="498" y="294" text-anchor="middle" font-size="12" fill="#333">A,120</text><text x="498" y="312" text-anchor="middle" font-size="12" fill="#333">A,90</text></g>
    <g><rect x="604" y="250" width="180" height="88" rx="8" fill="#FFFFFF" stroke="#DCE0E2" stroke-width="1.5"/><text x="694" y="272" text-anchor="middle" font-size="13" font-weight="700" fill="#1B5162" font-family="-apple-system,sans-serif">part3</text><text x="694" y="294" text-anchor="middle" font-size="12" fill="#333">C,80</text><text x="694" y="311" text-anchor="middle" font-size="12" fill="#333">C,140</text><text x="694" y="328" text-anchor="middle" font-size="12" fill="#333">B,110</text></g>
    <line x1="106" y1="338" x2="106" y2="366" stroke="#618794" stroke-width="2" marker-end="url(#mr-arrow)"/><line x1="302" y1="338" x2="302" y2="366" stroke="#618794" stroke-width="2" marker-end="url(#mr-arrow)"/><line x1="498" y1="338" x2="498" y2="366" stroke="#618794" stroke-width="2" marker-end="url(#mr-arrow)"/><line x1="694" y1="338" x2="694" y2="366" stroke="#618794" stroke-width="2" marker-end="url(#mr-arrow)"/>
    <path d="M40,368 L760,368 L784,398 L760,428 L40,428 L16,398 Z" fill="#DCE0E2" stroke="#618794" stroke-width="2"/><text x="400" y="403" text-anchor="middle" font-size="14" font-weight="700" fill="#1B5162" font-family="-apple-system,sans-serif">Shuffle: groupBy('id') co-locates rows by key</text>
    <line x1="240" y1="428" x2="240" y2="456" stroke="#618794" stroke-width="2" marker-end="url(#mr-arrow)"/><line x1="560" y1="428" x2="560" y2="456" stroke="#618794" stroke-width="2" marker-end="url(#mr-arrow)"/>
    <g><rect x="110" y="458" width="260" height="96" rx="8" fill="#FFFFFF" stroke="#618794" stroke-width="1.5"/><text x="240" y="480" text-anchor="middle" font-size="13" font-weight="700" fill="#1B5162" font-family="-apple-system,sans-serif">part0</text><text x="240" y="503" text-anchor="middle" font-size="12" fill="#333">A,100  A,120  A,90</text><text x="240" y="522" text-anchor="middle" font-size="12" fill="#333">B,75  B,110</text></g>
    <g><rect x="430" y="458" width="260" height="96" rx="8" fill="#FFFFFF" stroke="#618794" stroke-width="1.5"/><text x="560" y="480" text-anchor="middle" font-size="13" font-weight="700" fill="#1B5162" font-family="-apple-system,sans-serif">part1</text><text x="560" y="503" text-anchor="middle" font-size="12" fill="#333">C,200  C,60  C,80  C,140</text><text x="560" y="522" text-anchor="middle" font-size="12" fill="#333">D,150</text></g>
    <line x1="240" y1="554" x2="240" y2="582" stroke="#618794" stroke-width="2" marker-end="url(#mr-arrow)"/><line x1="560" y1="554" x2="560" y2="582" stroke="#618794" stroke-width="2" marker-end="url(#mr-arrow)"/>
    <rect x="16" y="584" width="768" height="56" rx="8" fill="#FFCC66" stroke="#CD7F32" stroke-width="2"/><text x="400" y="617" text-anchor="middle" font-size="14" font-weight="700" fill="#7a4a10" font-family="-apple-system,sans-serif">Reduce stage: agg sum('amt'), then write</text>
    <line x1="240" y1="640" x2="240" y2="668" stroke="#618794" stroke-width="2" marker-end="url(#mr-arrow)"/><line x1="560" y1="640" x2="560" y2="668" stroke="#618794" stroke-width="2" marker-end="url(#mr-arrow)"/>
    <g><rect x="110" y="670" width="260" height="80" rx="8" fill="#F9F7F4" stroke="#DCE0E2" stroke-width="1.5"/><text x="240" y="692" text-anchor="middle" font-size="13" font-weight="700" fill="#1B5162" font-family="-apple-system,sans-serif">part-00000.csv</text><text x="240" y="715" text-anchor="middle" font-size="12" fill="#333">A,310</text><text x="240" y="733" text-anchor="middle" font-size="12" fill="#333">B,185</text></g>
    <g><rect x="430" y="670" width="260" height="80" rx="8" fill="#F9F7F4" stroke="#DCE0E2" stroke-width="1.5"/><text x="560" y="692" text-anchor="middle" font-size="13" font-weight="700" fill="#1B5162" font-family="-apple-system,sans-serif">part-00001.csv</text><text x="560" y="715" text-anchor="middle" font-size="12" fill="#333">C,480</text><text x="560" y="733" text-anchor="middle" font-size="12" fill="#333">D,150</text></g>
    </g>
  </svg>
  </div>
</details>
<br/>
<details>
<summary style="cursor: pointer; list-style: none; user-select: none">
<div style="border-left: 4px solid #1B5162; background: transparent; padding: 16px 20px; border-radius: 4px; margin: 16px 0">
<div style="display: flex; align-items: center; gap: 12px">
<span>&#x25B6;</span>
<strong style="font-size: 1.1em; color: #1B5162">Expand for More Details</strong>
</div>
</div>
</summary>
<div style="border-left: 4px solid #1B5162; padding: 4px 24px 14px 24px; margin: -16px 0 16px 0">
<div style="font-size: 1.2em; font-weight: bold; color: #1B5162; margin: 12px 0 4px 0">Map Stage</div>
<ul style="font-size: 1.1em; line-height: 1.45; color: #333; margin: 0 0 10px 0; padding-left: 24px">
<li style="margin: 4px 0">Each input file becomes a partition, and the projection (<code>select</code>) and filter are applied to each partition independently - no data movement is required, so these are <em>narrow</em> transformations pipelined into one stage.</li>
<li style="margin: 4px 0">Note how <code>B,30</code> and <code>D,50</code> disappear at this stage: they fail the <code>amt &gt; 50</code> filter. Filtering early shrinks everything downstream, including the expensive shuffle.</li>
</ul>
<div style="font-size: 1.2em; font-weight: bold; color: #1B5162; margin: 14px 0 4px 0">Shuffle</div>
<ul style="font-size: 1.1em; line-height: 1.45; color: #333; margin: 0 0 10px 0; padding-left: 24px">
<li style="margin: 4px 0"><code>groupBy('id')</code> requires all rows with the same id to be in the same partition, so Spark redistributes the four map-side partitions into reduce-side partitions by hashing the key.</li>
</ul>
<div style="font-size: 1.2em; font-weight: bold; color: #1B5162; margin: 14px 0 4px 0">Reduce Stage</div>
<ul style="font-size: 1.1em; line-height: 1.45; color: #333; margin: 0 0 10px 0; padding-left: 24px">
<li style="margin: 4px 0">With each key's rows co-located, the sum is computed per key, and each reduce partition writes its own output file - which is why the result is a directory containing <code>part-00000.csv</code> and <code>part-00001.csv</code> rather than a single file.</li>
</ul>
</div>
</details>

## E. Spark's Implementation

MapReduce sits at the core of everything Spark does.

**Fundamental pattern:**

- Every Spark operation maps to Map / Shuffle / Reduce
- Even simple transformations follow this model
- It is the core building block for all distributed operations

**Common examples:**

| Operation | Map | Shuffle | Reduce |
|-----------|-----|---------|--------|
| __`groupBy`__ | Extract keys | Redistribute by key | Aggregate |
| __`join`__ | Prepare keys | Co-locate matching keys | Combine rows |
| __`filter`__ | Evaluate condition | Not required | Not required |

## Summary

- Spark uses shared-nothing execution: independent nodes process mutually exclusive partitions in parallel
- One partition = one task; partition count and size determine parallelism
- The shuffle redistributes data by key between stages and is the most expensive operation in Spark
- Every DataFrame pipeline compiles down to the MapReduce pattern: narrow transformations pipeline within a stage, wide transformations create shuffle boundaries

## Knowledge Check

Answer the questions below to check your understanding of this lecture.

<div style="font-family:-apple-system,BlinkMacSystemFont,Segoe UI,Roboto,sans-serif;width:100%;margin:10px 0;padding:24px;background:#f5f7fa;border-radius:12px;border:1px solid #e0e0e0;box-sizing:border-box">
<div style="display:flex;justify-content:space-between;align-items:center;margin-bottom:8px"><div><div style="font-size:1.4em;font-weight:600;color:#333">📝 Multiple Choice Quiz</div><div style="font-size:0.95em;color:#666;margin-top:4px">Knowledge Check</div></div><span style="padding:8px 16px;border-radius:20px;font-weight:600;font-size:0.9em;background:#e3f2fd;color:#1976d2">3 questions</span></div>
<div style="background:#fff;border-radius:12px;border:2px solid #e0e0e0;padding:18px 24px;margin:20px 0 12px 0"><div style="font-size:0.85em;font-weight:600;color:#1976d2;margin-bottom:6px">Question 1 of 3</div><div style="font-size:1.15em;color:#333;line-height:1.6">In Spark, one partition is processed by exactly one ____.</div></div>
<div style="display:flex;align-items:center;gap:16px;padding:12px 20px;background:#fff;border:2px solid #e0e0e0;border-radius:8px;margin:0 0 8px 0"><div style="width:32px;height:32px;min-width:32px;border:2px solid #e0e0e0;border-radius:50%;display:flex;align-items:center;justify-content:center;font-weight:700;color:#333">A</div><div style="flex:1;color:#333">Driver</div></div>
<div style="display:flex;align-items:center;gap:16px;padding:12px 20px;background:#fff;border:2px solid #e0e0e0;border-radius:8px;margin:0 0 8px 0"><div style="width:32px;height:32px;min-width:32px;border:2px solid #e0e0e0;border-radius:50%;display:flex;align-items:center;justify-content:center;font-weight:700;color:#333">B</div><div style="flex:1;color:#333">Task</div></div>
<div style="display:flex;align-items:center;gap:16px;padding:12px 20px;background:#fff;border:2px solid #e0e0e0;border-radius:8px;margin:0 0 8px 0"><div style="width:32px;height:32px;min-width:32px;border:2px solid #e0e0e0;border-radius:50%;display:flex;align-items:center;justify-content:center;font-weight:700;color:#333">C</div><div style="flex:1;color:#333">Worker</div></div>
<div style="display:flex;align-items:center;gap:16px;padding:12px 20px;background:#fff;border:2px solid #e0e0e0;border-radius:8px;margin:0 0 8px 0"><div style="width:32px;height:32px;min-width:32px;border:2px solid #e0e0e0;border-radius:50%;display:flex;align-items:center;justify-content:center;font-weight:700;color:#333">D</div><div style="flex:1;color:#333">Job</div></div>
<details style="margin:4px 0 8px 0"><summary style="cursor:pointer;color:#1976d2;font-weight:600;margin:4px 0">Show answer</summary><div style="margin-top:8px;padding:12px 16px;background:#e8f5e9;border:2px solid #4caf50;border-radius:8px"><div style="font-weight:700;color:#2e7d32">✓ B. Task</div><div style="color:#333;margin-top:6px;line-height:1.5">The unit of parallel work in Spark is the task, and each task processes exactly one partition. More partitions means more tasks that can run in parallel.</div></div></details>
<div style="background:#fff;border-radius:12px;border:2px solid #e0e0e0;padding:18px 24px;margin:20px 0 12px 0"><div style="font-size:0.85em;font-weight:600;color:#1976d2;margin-bottom:6px">Question 2 of 3</div><div style="font-size:1.15em;color:#333;line-height:1.6">Which of the following operations requires a shuffle?</div></div>
<div style="display:flex;align-items:center;gap:16px;padding:12px 20px;background:#fff;border:2px solid #e0e0e0;border-radius:8px;margin:0 0 8px 0"><div style="width:32px;height:32px;min-width:32px;border:2px solid #e0e0e0;border-radius:50%;display:flex;align-items:center;justify-content:center;font-weight:700;color:#333">A</div><div style="flex:1;color:#333">select()</div></div>
<div style="display:flex;align-items:center;gap:16px;padding:12px 20px;background:#fff;border:2px solid #e0e0e0;border-radius:8px;margin:0 0 8px 0"><div style="width:32px;height:32px;min-width:32px;border:2px solid #e0e0e0;border-radius:50%;display:flex;align-items:center;justify-content:center;font-weight:700;color:#333">B</div><div style="flex:1;color:#333">filter()</div></div>
<div style="display:flex;align-items:center;gap:16px;padding:12px 20px;background:#fff;border:2px solid #e0e0e0;border-radius:8px;margin:0 0 8px 0"><div style="width:32px;height:32px;min-width:32px;border:2px solid #e0e0e0;border-radius:50%;display:flex;align-items:center;justify-content:center;font-weight:700;color:#333">C</div><div style="flex:1;color:#333">groupBy() followed by an aggregation</div></div>
<div style="display:flex;align-items:center;gap:16px;padding:12px 20px;background:#fff;border:2px solid #e0e0e0;border-radius:8px;margin:0 0 8px 0"><div style="width:32px;height:32px;min-width:32px;border:2px solid #e0e0e0;border-radius:50%;display:flex;align-items:center;justify-content:center;font-weight:700;color:#333">D</div><div style="flex:1;color:#333">withColumn() with a simple arithmetic expression</div></div>
<details style="margin:4px 0 8px 0"><summary style="cursor:pointer;color:#1976d2;font-weight:600;margin:4px 0">Show answer</summary><div style="margin-top:8px;padding:12px 16px;background:#e8f5e9;border:2px solid #4caf50;border-radius:8px"><div style="font-weight:700;color:#2e7d32">✓ C. groupBy() followed by an aggregation</div><div style="color:#333;margin-top:6px;line-height:1.5">Grouping requires all rows with the same key to be co-located in the same partition, which forces an all-to-all redistribution of data - the shuffle. Projections and filters operate on each partition independently.</div></div></details>
<div style="background:#fff;border-radius:12px;border:2px solid #e0e0e0;padding:18px 24px;margin:20px 0 12px 0"><div style="font-size:0.85em;font-weight:600;color:#1976d2;margin-bottom:6px">Question 3 of 3</div><div style="font-size:1.15em;color:#333;line-height:1.6">In the MapReduce model, why is it beneficial to filter data during the map stage?</div></div>
<div style="display:flex;align-items:center;gap:16px;padding:12px 20px;background:#fff;border:2px solid #e0e0e0;border-radius:8px;margin:0 0 8px 0"><div style="width:32px;height:32px;min-width:32px;border:2px solid #e0e0e0;border-radius:50%;display:flex;align-items:center;justify-content:center;font-weight:700;color:#333">A</div><div style="flex:1;color:#333">Filters can only be evaluated before a shuffle</div></div>
<div style="display:flex;align-items:center;gap:16px;padding:12px 20px;background:#fff;border:2px solid #e0e0e0;border-radius:8px;margin:0 0 8px 0"><div style="width:32px;height:32px;min-width:32px;border:2px solid #e0e0e0;border-radius:50%;display:flex;align-items:center;justify-content:center;font-weight:700;color:#333">B</div><div style="flex:1;color:#333">It reduces the amount of data that must be shuffled across the network</div></div>
<div style="display:flex;align-items:center;gap:16px;padding:12px 20px;background:#fff;border:2px solid #e0e0e0;border-radius:8px;margin:0 0 8px 0"><div style="width:32px;height:32px;min-width:32px;border:2px solid #e0e0e0;border-radius:50%;display:flex;align-items:center;justify-content:center;font-weight:700;color:#333">C</div><div style="flex:1;color:#333">The reduce stage cannot remove rows</div></div>
<div style="display:flex;align-items:center;gap:16px;padding:12px 20px;background:#fff;border:2px solid #e0e0e0;border-radius:8px;margin:0 0 8px 0"><div style="width:32px;height:32px;min-width:32px;border:2px solid #e0e0e0;border-radius:50%;display:flex;align-items:center;justify-content:center;font-weight:700;color:#333">D</div><div style="flex:1;color:#333">Map-side filtering is required for fault tolerance</div></div>
<details style="margin:4px 0 8px 0"><summary style="cursor:pointer;color:#1976d2;font-weight:600;margin:4px 0">Show answer</summary><div style="margin-top:8px;padding:12px 16px;background:#e8f5e9;border:2px solid #4caf50;border-radius:8px"><div style="font-weight:700;color:#2e7d32">✓ B. It reduces the amount of data that must be shuffled across the network</div><div style="color:#333;margin-top:6px;line-height:1.5">The shuffle is the most expensive operation in Spark because data is serialized and moved over the network. Filtering early (filter early, filter often) shrinks the shuffle and everything downstream of it.</div></div></details>
</div>

&copy; <span id="dbx-year"></span> Databricks, Inc. All rights reserved. Apache, Apache Spark, Spark, the Spark Logo, Apache Iceberg, Iceberg, and the Apache Iceberg logo are trademarks of the <a href="https://www.apache.org/" target="_blank" style="font-size: 1em; color: #1B5162; text-decoration: underline">Apache Software Foundation</a>.<br/><br/><a href="https://databricks.com/privacy-policy" target="_blank" style="text-decoration: underline">Privacy Policy</a> | <a href="https://databricks.com/terms-of-use" target="_blank" style="text-decoration: underline">Terms of Use</a> | <a href="https://help.databricks.com/" target="_blank" style="text-decoration: underline">Support</a>

<script> document.getElementById("dbx-year").textContent = new Date().getFullYear(); </script>